## Lag Feature

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
df = pd.read_csv("preprocessed_energy_data.csv")

In [2]:
# Ensure proper time sorting using Time_Code
if "Time_Code" in df.columns:
    df["Time_Code"] = df["Time_Code"].astype(str)
    df["Time_Code"] = df["Time_Code"].str.extract("(\d+)").astype(float)

# Sort by Country_Name and Time_Code
if "Country_Name" in df.columns and "Time_Code" in df.columns:
    df = df.sort_values(["Country_Name", "Time_Code"])

# Create Lag Feature for Renewable Share
renewable_col = "Renewable_electricity_share_of_total_electricity_output_percent_[4.1_SHARE.RE.IN.ELECTRICITY]"

if renewable_col in df.columns:
    df["renewable_share_t_minus_1"] = (
        df.groupby("Country_Name")[renewable_col].shift(1)
    )

    # Create Growth Feature
    df["renewable_share_growth"] = (
        df.groupby("Country_Name")[renewable_col].pct_change()
    )

    # Validate Year Continuity
    df["year_diff"] = df.groupby("Country_Name")["Time_Code"].diff()

    # Keep only continuous yearly data
    df = df[(df["year_diff"].isna()) | (df["year_diff"] == 1)]

    # Drop first-year rows where lag is NaN
    df = df.dropna(subset=["renewable_share_t_minus_1"])

    # Remove helper column
    df = df.drop(columns=["year_diff"])

print("\nLag Features Added Successfully.")


Lag Features Added Successfully.
